In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("gpt-5-nano")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t water it today—I wasn’t in the office to do it. Would you like me to set up a reminder for the next watering, or log a note in the office maintenance plan so someone covers it? If you want me to water it next time I’m in, I can add that to my task list as well.


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [9]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Usually every 12–24 months, depending on how fast the plant is growing. If you start to notice signs, you might need to repot sooner. Look for:

- Roots circling inside the pot or visible from drainage holes
- Pot becoming root-bound or growth slowing despite good care
- Soil draining very slowly or staying soggy
- Pot feeling very light after watering, or the plant looks stressed

If growth has been healthy, aim for spring as a good time to repot. When you do repot, pick a pot 1–2 inches larger in diameter, use fresh potting mix with drainage, gently loosen roots, and water after planting.

Want me to check the current pot size and plant to estimate more precisely?


In [10]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07
